# Synthetic Dataset Creation

In [8]:
import os
import random
import torch
import gc
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, TorchAoConfig
from torchao.quantization import Int8WeightOnlyConfig

W0822 21:54:20.809000 14316 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [9]:
hf_token = os.getenv("HF_TOKEN")

models = {
    "LLAMA": "meta-llama/Llama-3.2-1B-Instruct",
    "PHI": "microsoft/Phi-4-mini-instruct",
    "GEMMA": "google/gemma-3-270m-it",
    "QWEN": "Qwen/Qwen3-4B-Instruct-2507"
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
COMPANIES = [
    ("Apex Robotics", "APXR"),
    ("Nova Semiconductor", "NVSC"),
    ("Vertex Cloud Systems", "VCS"),
    ("Quantum Dynamics", "QDYN"),
    ("Orion Technologies", "ORNT"),
]

def generate_ground_truth(id):
    company, ticker = random.choice(COMPANIES)

    year = random.randint(2023, 2026)
    quarter = random.choice(["Q1", "Q2", "Q3", "Q4"])

    # Financial values in millions of USD
    previous_revenue = round(random.uniform(100, 5000), 2)

    # Allow both growth and contraction
    revenue_growth = random.uniform(-0.20, 0.40)
    revenue = round(previous_revenue * (1 + revenue_growth), 2)

    # Profit margin can be negative
    net_margin = random.uniform(-0.10, 0.25)
    net_income = round(revenue * net_margin, 2)

    # Simplified diluted share count, in millions
    diluted_shares = round(random.uniform(20, 500), 2)
    eps = round(net_income / diluted_shares, 2)

    return {
        "id": id,
        "company": company,
        "ticker": ticker,
        "year": year,
        "quarter": quarter,
        "revenue_musd": revenue,
        "revenue_yoy_growth_pct": round(revenue_growth * 100, 1),
        "net_income_musd": net_income,
        "diluted_eps_usd": eps,
    }

In [11]:
ground_truth_examples = [generate_ground_truth("0" + str(i)) for i in range(10)]

ground_truth_examples

[{'id': '00',
  'company': 'Nova Semiconductor',
  'ticker': 'NVSC',
  'year': 2023,
  'quarter': 'Q1',
  'revenue_musd': 404.68,
  'revenue_yoy_growth_pct': -9.5,
  'net_income_musd': -0.17,
  'diluted_eps_usd': -0.0},
 {'id': '01',
  'company': 'Apex Robotics',
  'ticker': 'APXR',
  'year': 2025,
  'quarter': 'Q3',
  'revenue_musd': 3452.47,
  'revenue_yoy_growth_pct': -10.1,
  'net_income_musd': 164.02,
  'diluted_eps_usd': 0.33},
 {'id': '02',
  'company': 'Apex Robotics',
  'ticker': 'APXR',
  'year': 2023,
  'quarter': 'Q1',
  'revenue_musd': 4849.43,
  'revenue_yoy_growth_pct': 36.0,
  'net_income_musd': -129.29,
  'diluted_eps_usd': -1.42},
 {'id': '03',
  'company': 'Nova Semiconductor',
  'ticker': 'NVSC',
  'year': 2024,
  'quarter': 'Q3',
  'revenue_musd': 2875.54,
  'revenue_yoy_growth_pct': -1.9,
  'net_income_musd': -208.05,
  'diluted_eps_usd': -0.75},
 {'id': '04',
  'company': 'Orion Technologies',
  'ticker': 'ORNT',
  'year': 2025,
  'quarter': 'Q2',
  'revenue_musd

In [12]:
def build_prompt(record):
    return f"""
Write a short fictional quarterly earnings announcement using the financial
data provided below.

Financial data:
- Company: {record["company"]}
- Ticker: {record["ticker"]}
- Reporting period: {record["quarter"]} {record["year"]}
- Revenue: ${record["revenue_musd"]} million
- Revenue year-over-year growth: {record["revenue_yoy_growth_pct"]}%
- Net income: ${record["net_income_musd"]} million
- Diluted EPS: ${record["diluted_eps_usd"]}

Requirements:
- Include all of the financial information above.
- Preserve all numerical values exactly.
- Do not invent additional financial figures.
- Write in the style of a concise corporate earnings announcement.
- Output only the announcement.
""".strip()

def generate_synth_example(model_id: str, ground_truth: dict):
    prompt = build_prompt(ground_truth)

    quant_config = TorchAoConfig(quant_type=Int8WeightOnlyConfig())

    messages = [
        {"role": "user", "content": prompt},
    ]

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    llm = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        dtype=torch.bfloat16,
        quantization_config=quant_config,
    )

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True,
    ).to(device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=1000,
        pad_token_id=tokenizer.pad_token_id,
    )

    input_length = inputs["input_ids"].shape[-1]
    generated_tokens = outputs[0, input_length:]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    del llm, inputs, outputs, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return text

In [13]:
def generate_synth_dataset_multimodel(models, ground_truth_examples):
    dataset = []
    for model in models:
        for example in ground_truth_examples:
            synth_example = generate_synth_example(models[model], example)
            dataset.append({
                "id": example["id"] + "_" + model,
                "ground_truth_id": example["id"],
                "generator_model": model,
                "announcement": synth_example,
                "ground_truth": example,
            })
            
    return pd.DataFrame(dataset)

In [14]:
synth_dataset = generate_synth_dataset_multimodel(models, ground_truth_examples)

W0822 21:54:21.830000 14316 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

... (11 outputs left out)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [20]:
synth_dataset.tail()

,id,ground_truth_id,generator_model,announcement,ground_truth
35,05_QWEN,05,QWEN,ORION TECHNOLOGIES (ORNT) ANNOUNCES Q1 2025 EA...,"{'id': '05', 'company': 'Orion Technologies', ..."
36,06_QWEN,06,QWEN,Vertex Cloud Systems (VCS) Reports Strong Q4 2...,"{'id': '06', 'company': 'Vertex Cloud Systems'..."
37,07_QWEN,07,QWEN,APX Robotics (APXR) Reports Q2 2025 Earnings ...,"{'id': '07', 'company': 'Apex Robotics', 'tick..."
38,08_QWEN,08,QWEN,APX Robotics (APXR) Reports Strong Q4 2026 Ear...,"{'id': '08', 'company': 'Apex Robotics', 'tick..."
39,09_QWEN,09,QWEN,Vertex Cloud Systems (VCS) Announces Q4 2023 E...,"{'id': '09', 'company': 'Vertex Cloud Systems'..."
